## Member 2 — Duplicate Records, Data Consistency & Medal-Counting Rules.

### Data Cleaning

### 1. Duplicate Records Investigation
### 2. Data Consistency Investigation
### 3. Medal-Counting Rules

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

athletes = pd.read_csv("../data/raw/athlete_events.csv")
regions = pd.read_csv("../data/raw/noc_regions.csv")

df = athletes.merge(regions, on="NOC", how="left")

In [2]:
# Number of exact duplicate records
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)
print("Percentage:", round(df.duplicated().mean() * 100, 2), "%")

Number of duplicate records: 1385
Percentage: 0.51 %


In [3]:
duplicates = df[df.duplicated(keep=False)]

duplicates.head(10)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes
1251,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,Belgium,NaN
1252,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,Belgium,NaN
4281,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4282,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4283,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4861,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Designs F...",NaN,Germany,NaN
4862,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Designs F...",NaN,Germany,NaN
4863,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN,Germany,NaN
4864,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN,Germany,NaN
4866,2777,Hermann Reinhard Alker,M,51.0,NaN,NaN,Germany,GER,1936 Summer,1936,Summer,Berlin,Art Competitions,"Art Competitions Mixed Architecture, Unknown E...",NaN,Germany,NaN


This is a different type of duplication.
An athlete can participate in multiple Olympic editions and multiple events. We need to distinguish legitimate repeated participation from suspicious repeated records.

In [4]:
# Check repeated athlete participation in the same event
duplicate_participations = df[
    df.duplicated(
        subset=["ID", "Games", "Event"],
        keep=False
    )
].sort_values(["ID", "Games", "Event"])

print("Repeated athlete-event records:",
      len(duplicate_participations))

duplicate_participations.head(20)

Repeated athlete-event records: 2167


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes
1251,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,Belgium,NaN
1252,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,Belgium,NaN
4017,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Silver,France,NaN
4018,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze,France,NaN
4281,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4282,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4283,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4863,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN,Germany,NaN
4864,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN,Germany,NaN
4861,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Designs F...",NaN,Germany,NaN


If the same athlete appears more than once in the same event and edition, we need to investigate why.

In [5]:
#compare duplicates
print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Duplicate athlete-event combinations:",
    df.duplicated(
        subset=["ID", "Games", "Event"]
    ).sum()
)

Exact duplicate rows: 1385
Duplicate athlete-event combinations: 1480


In [6]:
duplicate_participations[
    ["ID", "Name", "Games", "Event", "Team", "NOC", "Medal"]
].head(30)

,ID,Name,Games,Event,Team,NOC,Medal
1251,704,Dsir Antoine Acket,1932 Summer,"Art Competitions Mixed Painting, Unknown Event",Belgium,BEL,NaN
1252,704,Dsir Antoine Acket,1932 Summer,"Art Competitions Mixed Painting, Unknown Event",Belgium,BEL,NaN
4017,2317,Auguste Albert,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,FRA,Silver
4018,2317,Auguste Albert,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,FRA,Bronze
4281,2449,William Truman Aldrich,1928 Summer,"Art Competitions Mixed Painting, Drawings And ...",United States,USA,NaN
4282,2449,William Truman Aldrich,1928 Summer,"Art Competitions Mixed Painting, Drawings And ...",United States,USA,NaN
4283,2449,William Truman Aldrich,1928 Summer,"Art Competitions Mixed Painting, Drawings And ...",United States,USA,NaN
4863,2777,Hermann Reinhard Alker,1928 Summer,"Art Competitions Mixed Architecture, Architect...",Germany,GER,NaN
4864,2777,Hermann Reinhard Alker,1928 Summer,"Art Competitions Mixed Architecture, Architect...",Germany,GER,NaN
4861,2777,Hermann Reinhard Alker,1928 Summer,"Art Competitions Mixed Architecture, Designs F...",Germany,GER,NaN


In [7]:
medal_variations = (
    duplicate_participations
    .groupby(["ID", "Games", "Event"])["Medal"]
    .nunique()
    .reset_index(name="unique_medals")
)

medal_variations = medal_variations[
    medal_variations["unique_medals"] > 1
]

print("Groups with different medal values:", len(medal_variations))
medal_variations.head(10)

Groups with different medal values: 21


,ID,Games,Event,unique_medals
1,2317,1900 Summer,Sailing Mixed 1-2 Ton,2
29,8683,1900 Summer,Sailing Mixed 0.5-1 Ton,2
119,26557,1900 Summer,Sailing Mixed 1-2 Ton,2
120,26559,1900 Summer,Sailing Mixed 1-2 Ton,2
121,26560,1900 Summer,Sailing Mixed 1-2 Ton,2
136,28501,1948 Summer,"Art Competitions Mixed Painting, Applied Arts",2
150,30274,1900 Summer,Sailing Mixed 3-10 Ton,2
151,30285,1900 Summer,Sailing Mixed 3-10 Ton,2
162,30927,1900 Summer,Sailing Mixed 1-2 Ton,2
214,39839,1900 Summer,Sailing Mixed 0-0.5 Ton,2


In [8]:
df.merge(
    medal_variations[["ID", "Games", "Event"]],
    on=["ID", "Games", "Event"],
    how="inner"
).sort_values(["ID", "Games", "Event"])[
    ["ID", "Name", "Games", "Event", "Team", "NOC", "Medal"]
].head(30)

,ID,Name,Games,Event,Team,NOC,Medal
0,2317,Auguste Albert,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,FRA,Silver
1,2317,Auguste Albert,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,FRA,Bronze
2,8683,Jacques Baudrier,1900 Summer,Sailing Mixed 0.5-1 Ton,Crabe II-1,FRA,Silver
3,8683,Jacques Baudrier,1900 Summer,Sailing Mixed 0.5-1 Ton,Crabe II-4,FRA,Bronze
4,26557,Bernard Alexandre Georges Edmond de Pourtals,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Gold
5,26557,Bernard Alexandre Georges Edmond de Pourtals,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Silver
6,26559,Hlne de Pourtals (Barbey-),1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Gold
7,26559,Hlne de Pourtals (Barbey-),1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Silver
8,26560,Hermann Alexandre de Pourtals,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Gold
9,26560,Hermann Alexandre de Pourtals,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Silver


In [9]:
# Extract exact duplicate rows
exact_duplicates = df[
    df.duplicated(keep=False)
].sort_values(["ID", "Games", "Event"])

print("Rows involved in exact duplicates:", len(exact_duplicates))

exact_duplicates.head(20)

Rows involved in exact duplicates: 1997


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes
1251,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,Belgium,NaN
1252,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,Belgium,NaN
4281,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4282,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4283,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN,USA,NaN
4863,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN,Germany,NaN
4864,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN,Germany,NaN
4861,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Designs F...",NaN,Germany,NaN
4862,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Designs F...",NaN,Germany,NaN
4866,2777,Hermann Reinhard Alker,M,51.0,NaN,NaN,Germany,GER,1936 Summer,1936,Summer,Berlin,Art Competitions,"Art Competitions Mixed Architecture, Unknown E...",NaN,Germany,NaN


In [10]:
athletes.duplicated().sum()

np.int64(1385)

In [11]:
print("Duplicated NOC codes:", regions["NOC"].duplicated().sum())

Duplicated NOC codes: 0


This means we can investigate removing exact duplicates without worrying that the merge caused them.

In [12]:
print("Original dataset shape:", df.shape)

df_no_exact_duplicates = df.drop_duplicates()

print("Shape after removing exact duplicates:",
      df_no_exact_duplicates.shape)

print("Rows removed:",
      len(df) - len(df_no_exact_duplicates))

Original dataset shape: (271116, 17)
Shape after removing exact duplicates: (269731, 17)
Rows removed: 1385


We can safely remove rows that are identical across all columns. We will not remove repeated athlete-event combinations just because their ID, Games, and Event match.

In [13]:
df_clean = df.drop_duplicates().copy()

print("Shape before cleaning:", df.shape)
print("Shape after cleaning:", df_clean.shape)
print("Remaining exact duplicates:", df_clean.duplicated().sum())

Shape before cleaning: (271116, 17)
Shape after cleaning: (269731, 17)
Remaining exact duplicates: 0


### Exact Duplicate Removal

The dataset initially contained 1,385 exact duplicate rows, representing approximately 0.51% of the original records.

We confirmed that these duplicates already existed in the original `athlete_events.csv` dataset and were not introduced by merging with the NOC regions table.

**Cleaning decision:** Remove exact duplicate rows using `drop_duplicates()`.

- Original shape: 271,116 rows × 17 columns
- Cleaned shape: 269,731 rows × 17 columns
- Rows removed: 1,385

Repeated athlete-event combinations with different medal values were preserved for further investigation because they may represent legitimate historical records.

In [14]:
# Numerical columns summary
df_clean[["Age", "Height", "Weight"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,260416.0,25.454776,6.163869,10.0,21.0,24.0,28.0,97.0
Height,210917.0,175.338953,10.518507,127.0,168.0,175.0,183.0,226.0
Weight,208204.0,70.701778,14.349027,25.0,60.0,70.0,79.0,214.0


In [15]:
# Inspect unusually low and high values
for col in ["Age", "Height", "Weight"]:
    print(f"\n--- {col} ---")

    print("Lowest values:")
    print(df_clean[col].nsmallest(10))

    print("\nHighest values:")
    print(df_clean[col].nlargest(10))


--- Age ---
Lowest values:
142882    10.0
43468     11.0
73461     11.0
79024     11.0
94058     11.0
101378    11.0
102916    11.0
140650    11.0
152798    11.0
237141    11.0
Name: Age, dtype: float64

Highest values:
257054    97.0
98118     96.0
60861     88.0
9371      84.0
154855    81.0
236912    81.0
138812    80.0
170993    77.0
56599     76.0
62839     76.0
Name: Age, dtype: float64

--- Height ---
Lowest values:
29333     127.0
29334     127.0
29335     127.0
29336     127.0
29337     127.0
29338     127.0
164821    127.0
180615    128.0
159934    130.0
220270    130.0
Name: Height, dtype: float64

Highest values:
265040    226.0
265041    226.0
265042    226.0
32376     223.0
207373    223.0
207374    223.0
207375    223.0
17669     221.0
59371     221.0
59372     221.0
Name: Height, dtype: float64

--- Weight ---
Lowest values:
40849     25.0
40850     25.0
40851     25.0
40852     25.0
40853     25.0
40854     25.0
109863    28.0
109864    28.0
109865    28.0
109866    2

my observations:
- Age: Age 10 is plausible for some historical Olympic participants. Age 97 is highly suspicious and needs investigation.
- Height: 127–226 cm is an unusual but potentially legitimate range.
- Weight: 25–214 kg is also unusual but not automatically invalid.

In [16]:
# Inspect athletes aged 80 or above
df_clean[df_clean["Age"] >= 80][
    ["Name", "Sex", "Age", "Sport", "Event", "Games", "Team"]
].sort_values("Age", ascending=False)

,Name,Sex,Age,Sport,Event,Games,Team
257054,John Quincy Adams Ward,M,97.0,Art Competitions,"Art Competitions Mixed Sculpturing, Statues",1928 Summer,United States
98118,Winslow Homer,M,96.0,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",1932 Summer,United States
60861,Thomas Cowperthwait Eakins,M,88.0,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",1932 Summer,United States
9371,George Denholm Armour,M,84.0,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",1948 Summer,Great Britain
154855,Robert Tait McKenzie,M,81.0,Art Competitions,"Art Competitions Mixed Sculpturing, Unknown Event",1948 Summer,Canada
236912,Louis Tauzin,M,81.0,Art Competitions,Art Competitions Mixed Sculpturing,1924 Summer,France
138812,Max Liebermann,M,80.0,Art Competitions,"Art Competitions Mixed Painting, Graphic Arts",1928 Summer,Germany


In [17]:
age_ranges = {
    "Under 15": df_clean["Age"] < 15,
    "15–30": df_clean["Age"].between(15, 30),
    "31–50": df_clean["Age"].between(31, 50),
    "51–70": df_clean["Age"].between(51, 70),
    "Over 70": df_clean["Age"] > 70
}

for label, condition in age_ranges.items():
    print(f"{label}: {condition.sum()}")

Under 15: 1074
15–30: 218307
31–50: 39559
51–70: 1415
Over 70: 61


The largest group is 15–30, representing the typical age range for Olympic athletes.
There are 1,074 records for athletes under 15, which may be legitimate, particularly in historical competitions.
There are 61 records above 70, which deserve individual inspection.
The age distribution is strongly concentrated in younger athletes, with a small older-age tail.

In [18]:
old_athletes = df_clean[df_clean["Age"] > 70][
    ["Name", "Sex", "Age", "Sport", "Event", "Games", "Team"]
].sort_values("Age", ascending=False)

print("Records above 70:", len(old_athletes))
old_athletes

Records above 70: 61


,Name,Sex,Age,Sport,Event,Games,Team
257054,John Quincy Adams Ward,M,97.0,Art Competitions,"Art Competitions Mixed Sculpturing, Statues",1928 Summer,United States
98118,Winslow Homer,M,96.0,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",1932 Summer,United States
60861,Thomas Cowperthwait Eakins,M,88.0,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",1932 Summer,United States
9371,George Denholm Armour,M,84.0,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",1948 Summer,Great Britain
154855,Robert Tait McKenzie,M,81.0,Art Competitions,"Art Competitions Mixed Sculpturing, Unknown Event",1948 Summer,Canada
...,...,...,...,...,...,...,...
150753,Charles William Martin,M,71.0,Sailing,Sailing Mixed Open,1900 Summer,Crabe II-12
150755,Charles William Martin,M,71.0,Sailing,Sailing Mixed 0.5-1 Ton,1900 Summer,Crabe II-1
150754,Charles William Martin,M,71.0,Sailing,Sailing Mixed Open,1900 Summer,Pirouette-31
215182,Thomas Foster Scott,M,71.0,Archery,Archery Men's Double American Round,1904 Summer,United States


These are associated with Art Competitions, which historically allowed participants outside the typical athletic age range. Other older participants appear in sports such as Sailing and Archery.

### Age Consistency Investigation

The age distribution revealed 61 records for participants older than 70 and 1,074 records for participants younger than 15.

Further inspection of older participants showed that several extreme ages belong to historical Art Competitions, including participants aged 97, 96, and 88. Other older participants were recorded in disciplines such as Sailing and Archery.

**Cleaning decision:** Retain these records because unusual ages are not sufficient evidence of invalid data. They may represent legitimate historical participation.

No age records were removed during this investigation.

In [19]:
# Missing values in numerical columns
numerical_cols = ["Age", "Height", "Weight"]

missing_summary = pd.DataFrame({
    "Missing Count": df_clean[numerical_cols].isna().sum(),
    "Missing Percentage": (
        df_clean[numerical_cols].isna().mean() * 100
    ).round(2)
})

missing_summary

,Missing Count,Missing Percentage
Age,9315,3.45
Height,58814,21.80
Weight,61527,22.81


Define reliable medal-counting rules
The most important distinction is between:
1. Athlete-event records: One row represents an athlete's participation in an event.
2. Individual medals: A medal awarded to an individual athlete.
3. Team medals: One medal awarded to a team, even if multiple athletes appear in the dataset as medal recipients.

In [20]:
# Medal distribution
df_clean["Medal"].value_counts(dropna=False)

Medal
NaN       229959
Gold       13369
Bronze     13295
Silver     13108
Name: count, dtype: int64

In [21]:
# Number of records by medal and sport
medal_by_sport = pd.crosstab(
    df_clean["Sport"],
    df_clean["Medal"]
)

medal_by_sport.sort_values(
    "Gold",
    ascending=False
).head(15)

Medal,Bronze,Gold,Silver
Sport,,,
Athletics,1296,1339,1334
Swimming,956,1099,993
Rowing,990,978,977
Gymnastics,719,791,746
Fencing,566,594,583
Hockey,512,518,498
Football,543,515,513
Ice Hockey,507,508,515
Sailing,367,447,407


In [22]:
#team events
medal_records = df_clean[df_clean["Medal"].notna()]

team_event_summary = (
    medal_records
    .groupby(["Games", "Event", "Team", "Medal"])
    .agg(
        athlete_count=("ID", "nunique"),
        record_count=("ID", "size")
    )
    .reset_index()
)

team_event_summary.sort_values(
    "athlete_count",
    ascending=False
).head(20)

,Games,Event,Team,Medal,athlete_count,record_count
1056,1908 Summer,Gymnastics Men's Team All-Around,Sweden,Gold,38,38
1055,1908 Summer,Gymnastics Men's Team All-Around,Norway,Silver,30,30
1378,1912 Summer,"Gymnastics Men's Team All-Around, Swedish System",Denmark,Silver,28,28
1767,1920 Summer,Gymnastics Men's Team All-Around,Italy,Gold,27,27
1766,1920 Summer,Gymnastics Men's Team All-Around,France,Bronze,26,26
1054,1908 Summer,Gymnastics Men's Team All-Around,Finland,Bronze,26,26
1771,1920 Summer,"Gymnastics Men's Team All-Around, Swedish System",Denmark,Silver,25,25
1769,1920 Summer,"Gymnastics Men's Team All-Around, Free System",Norway,Silver,25,25
14433,2004 Summer,Baseball Men's Baseball,Australia,Silver,24,24
14435,2004 Summer,Baseball Men's Baseball,Japan,Bronze,24,24


- Sweden — Gymnastics Men's Team All-Around, 1908: 38 athlete records, Gold.
- Norway — same event: 30 athlete records, Silver.
- Denmark — Gymnastics Team event, 1912: 28 athlete records, Silver.
If we count every row, Sweden would appear to have won 38 Gold medals in that event. In reality, those records represent one team medal.

However, we cannot simply count one medal per country per event either. A country can have multiple athletes winning medals in an individual event, or multiple boats competing in a Sailing event.

We need to distinguish two types of medal awards.
Event type	Counting rule
Individual events	Count each distinct athlete's medal
Team events	Count each distinct medal-winning team once
Tied positions	Preserve each distinct medal award
Exact duplicate records	Remove before counting
Missing medals	Exclude from medal tallies

For individual events, a useful key is:
ID + Games + Event + Medal

For team events, a candidate key is:
Games + Event + Team + NOC + Medal

The Team field is important because it can distinguish separate boats or team entries belonging to the same country.

In [23]:
# Events with multiple athletes sharing the same team medal
multi_athlete_teams = team_event_summary[
    team_event_summary["athlete_count"] > 1
].sort_values("athlete_count", ascending=False)

multi_athlete_teams[
    ["Games", "Event", "Team", "Medal", "athlete_count"]
].head(30)

,Games,Event,Team,Medal,athlete_count
1056,1908 Summer,Gymnastics Men's Team All-Around,Sweden,Gold,38
1055,1908 Summer,Gymnastics Men's Team All-Around,Norway,Silver,30
1378,1912 Summer,"Gymnastics Men's Team All-Around, Swedish System",Denmark,Silver,28
1767,1920 Summer,Gymnastics Men's Team All-Around,Italy,Gold,27
1054,1908 Summer,Gymnastics Men's Team All-Around,Finland,Bronze,26
1766,1920 Summer,Gymnastics Men's Team All-Around,France,Bronze,26
1771,1920 Summer,"Gymnastics Men's Team All-Around, Swedish System",Denmark,Silver,25
1769,1920 Summer,"Gymnastics Men's Team All-Around, Free System",Norway,Silver,25
17793,2014 Winter,Ice Hockey Men's Ice Hockey,Finland,Bronze,24
17792,2014 Winter,Ice Hockey Men's Ice Hockey,Canada,Gold,24


In [24]:
#see unique values
multi_athlete_teams["Event"].drop_duplicates().head(40).tolist()

["Gymnastics Men's Team All-Around",
 "Gymnastics Men's Team All-Around, Swedish System",
 "Gymnastics Men's Team All-Around, Free System",
 "Ice Hockey Men's Ice Hockey",
 "Baseball Men's Baseball",
 "Hockey Men's Hockey",
 "Handball Men's Handball",
 "Ice Hockey Women's Ice Hockey",
 "Football Men's Football",
 "Rugby Men's Rugby",
 "Football Women's Football",
 "Hockey Women's Hockey",
 "Rowing Men's 17-Man Naval Rowing Boats",
 "Handball Women's Handball",
 "Softball Women's Softball",
 "Basketball Men's Basketball",
 "Water Polo Men's Water Polo",
 "Water Polo Women's Water Polo",
 "Rugby Sevens Men's Rugby Sevens",
 "Basketball Women's Basketball",
 "Volleyball Men's Volleyball",
 "Gymnastics Women's Team All-Around",
 "Rugby Sevens Women's Rugby Sevens",
 "Cycling Men's Road Race, Team",
 "Volleyball Women's Volleyball",
 "Lacrosse Men's Lacrosse",
 'Sailing Mixed 8 metres',
 'Alpinism Mixed Alpinism',
 "Cricket Men's Cricket",
 "Athletics Men's Cross-Country, Team",
 "Rowing Wo

- In an individual Athletics event, two athletes from the same country may both win medals. We must count both.
- In a team event, 24 athletes representing one team may share one medal. We must count one.
- In Sailing, the Team field can identify separate boats, so we must not combine different boats just because they represent the same country.
Therefore, we need to distinguish individual events from team events before calculating country medal totals.

In [25]:
team_keywords = [
    "Team",
    "Relay",
    "Doubles",
    "Pairs",
    "Coxed",
    "Eight",
    "Four",
    "Football",
    "Basketball",
    "Hockey",
    "Volleyball",
    "Water Polo",
    "Baseball",
    "Softball"
]

team_like_events = sorted(
    event for event in df_clean["Event"].dropna().unique()
    if any(keyword.lower() in event.lower()
           for keyword in team_keywords)
)

print("Potential team-related events:", len(team_like_events))

for event in team_like_events[:60]:
    print(event)

Potential team-related events: 250
Archery Men's Pole Archery, Large Birds, Team
Archery Men's Pole Archery, Small Birds, Team
Archery Men's Target Archery, 28 metres, Team
Archery Men's Target Archery, 33 metres, Team
Archery Men's Target Archery, 50 metres, Team
Archery Men's Team
Archery Men's Team Round
Archery Women's Team
Archery Women's Team Round
Athletics Men's 1,600 metres Medley Relay
Athletics Men's 3 mile, Team
Athletics Men's 3,000 metres, Team
Athletics Men's 4 mile, Team
Athletics Men's 4 x 100 metres Relay
Athletics Men's 4 x 400 metres Relay
Athletics Men's 5,000 metres, Team
Athletics Men's 56-pound Weight Throw
Athletics Men's Cross-Country, Team
Athletics Women's 4 x 100 metres Relay
Athletics Women's 4 x 400 metres Relay
Badminton Men's Doubles
Badminton Mixed Doubles
Badminton Women's Doubles
Baseball Men's Baseball
Basketball Men's Basketball
Basketball Women's Basketball
Basque Pelota Men's Two-Man Teams With Cesta
Beach Volleyball Men's Beach Volleyball
Beach 

need a more reliable classification than simply searching event names.

In [26]:
strong_team_keywords = ["Team", "Relay", "Doubles", "Pairs"]

strong_team_events = sorted([
    event for event in df_clean["Event"].dropna().unique()
    if any(keyword.lower() in event.lower()
           for keyword in strong_team_keywords)
])

print("Potential team/pair events:", len(strong_team_events))

for event in strong_team_events:
    print(event)

Potential team/pair events: 128
Archery Men's Pole Archery, Large Birds, Team
Archery Men's Pole Archery, Small Birds, Team
Archery Men's Target Archery, 28 metres, Team
Archery Men's Target Archery, 33 metres, Team
Archery Men's Target Archery, 50 metres, Team
Archery Men's Team
Archery Men's Team Round
Archery Women's Team
Archery Women's Team Round
Athletics Men's 1,600 metres Medley Relay
Athletics Men's 3 mile, Team
Athletics Men's 3,000 metres, Team
Athletics Men's 4 mile, Team
Athletics Men's 4 x 100 metres Relay
Athletics Men's 4 x 400 metres Relay
Athletics Men's 5,000 metres, Team
Athletics Men's Cross-Country, Team
Athletics Women's 4 x 100 metres Relay
Athletics Women's 4 x 400 metres Relay
Badminton Men's Doubles
Badminton Mixed Doubles
Badminton Women's Doubles
Basque Pelota Men's Two-Man Teams With Cesta
Biathlon Men's 4 x 7.5 kilometres Relay
Biathlon Mixed 2 x 6 kilometres and 2 x 7.5 kilometres Relay
Biathlon Women's 3 x 7.5 kilometres Relay
Biathlon Women's 4 x 6 kil

In [27]:
team_event_medals = medal_records[
    medal_records["Event"].isin(strong_team_events)
]

print("Medal-winning athlete records:", len(team_event_medals))

print(
    "Unique events:",
    team_event_medals["Event"].nunique()
)

print(
    "Unique athletes:",
    team_event_medals["ID"].nunique()
)

Medal-winning athlete records: 10841
Unique events: 128
Unique athletes: 8181


inspect how athletes are grouped within each medal-winning team

If a team wins one medal and has 20 athletes, the dataset may contain 20 athlete records for that single medal award. We also want to identify cases where the same country has multiple team entries in the same event.

In [28]:
candidate_medals = medal_records[
    medal_records["Event"].isin(strong_team_events)
].copy()

team_medal_summary = (
    candidate_medals
    .groupby(["Games", "Event", "NOC", "Team", "Medal"])
    .agg(
        athlete_count=("ID", "nunique"),
        record_count=("ID", "size"),
        region=("region", "first")
    )
    .reset_index()
    .sort_values("athlete_count", ascending=False)
)

team_medal_summary.head(20)

,Games,Event,NOC,Team,Medal,athlete_count,record_count,region
112,1908 Summer,Gymnastics Men's Team All-Around,SWE,Sweden,Gold,38,38,Sweden
111,1908 Summer,Gymnastics Men's Team All-Around,NOR,Norway,Silver,30,30,Norway
178,1912 Summer,"Gymnastics Men's Team All-Around, Swedish System",DEN,Denmark,Silver,28,28,Denmark
273,1920 Summer,Gymnastics Men's Team All-Around,ITA,Italy,Gold,27,27,Italy
110,1908 Summer,Gymnastics Men's Team All-Around,FIN,Finland,Bronze,26,26,Finland
272,1920 Summer,Gymnastics Men's Team All-Around,FRA,France,Bronze,26,26,France
275,1920 Summer,"Gymnastics Men's Team All-Around, Free System",NOR,Norway,Silver,25,25,Norway
277,1920 Summer,"Gymnastics Men's Team All-Around, Swedish System",DEN,Denmark,Silver,25,25,Denmark
177,1912 Summer,"Gymnastics Men's Team All-Around, Free System",NOR,Norway,Gold,24,24,Norway
180,1912 Summer,"Gymnastics Men's Team All-Around, Swedish System",SWE,Sweden,Gold,24,24,Sweden


Each athlete has an individual row, but all 38 Swedish athletes received the same team gold medal. Counting rows would therefore count one team medal 38 times.

The same pattern appears in other editions and events, such as the 1912 and 1920 gymnastics team competitions.

our 128 potential team/pair events are a good starting point, but remember:
- Team events: one medal award per medal-winning team.
- Pairs/doubles: one medal award per pair, even if two athletes appear in the dataset.
- Individual events: one medal award per medal-winning athlete.
- Ties: preserve every medal awarded.
Also, Team is not always the country name. It can represent a boat or another team-entry name, particularly in historical sailing events. Use NOC for the country attribution, while retaining Team when distinguishing separate entries.

In [29]:
#Before defining our final counting rules, let's inspect a doubles event
doubles_medals = medal_records[
    medal_records["Event"].str.contains(
        "Doubles|Pairs",
        case=False,
        na=False
    )
]

doubles_summary = (
    doubles_medals
    .groupby(["Games", "Event", "NOC", "Team", "Medal"])
    .agg(
        athlete_count=("ID", "nunique"),
        record_count=("ID", "size")
    )
    .reset_index()
)

doubles_summary.sort_values(
    "athlete_count",
    ascending=False
).head(20)

,Games,Event,NOC,Team,Medal,athlete_count,record_count
8,1900 Summer,Rowing Men's Coxed Pairs,NED,Minerva Amsterdam,Gold,3,3
6,1900 Summer,Rowing Men's Coxed Pairs,FRA,Rowing Club Castillon-3,Bronze,3,3
453,1992 Summer,Rowing Men's Coxed Pairs,ROU,Romania,Bronze,3,3
452,1992 Summer,Rowing Men's Coxed Pairs,ITA,Italy,Silver,3,3
451,1992 Summer,Rowing Men's Coxed Pairs,GBR,Great Britain,Gold,3,3
398,1988 Summer,Rowing Men's Coxed Pairs,ITA,Italy,Gold,3,3
397,1988 Summer,Rowing Men's Coxed Pairs,GDR,East Germany,Silver,3,3
396,1988 Summer,Rowing Men's Coxed Pairs,GBR,Great Britain,Bronze,3,3
338,1980 Summer,Rowing Men's Coxed Pairs,YUG,Yugoslavia,Bronze,3,3
337,1980 Summer,Rowing Men's Coxed Pairs,URS,Soviet Union,Silver,3,3


We should verify the opposite case: an individual event where each medal-winning athlete should count as one medal award.

In [30]:
individual_medal_records = medal_records[
    medal_records["Event"].str.contains(
        "100 metres Freestyle",
        case=False,
        na=False
    )
]

individual_summary = (
    individual_medal_records
    .groupby(["Games", "Event", "NOC", "Team", "Medal"])
    .agg(
        athlete_count=("ID", "nunique"),
        record_count=("ID", "size")
    )
    .reset_index()
)

individual_summary.head(20)

,Games,Event,NOC,Team,Medal,athlete_count,record_count
0,1896 Summer,Swimming Men's 100 metres Freestyle,AUT,Austria,Silver,1,1
1,1896 Summer,Swimming Men's 100 metres Freestyle,HUN,Hungary,Gold,1,1
2,1896 Summer,Swimming Men's 100 metres Freestyle For Sailors,GRE,Greece,Bronze,1,1
3,1896 Summer,Swimming Men's 100 metres Freestyle For Sailors,GRE,Greece,Gold,1,1
4,1896 Summer,Swimming Men's 100 metres Freestyle For Sailors,GRE,Greece,Silver,1,1
5,1906 Summer,Swimming Men's 100 metres Freestyle,AUS,Australia,Bronze,1,1
6,1906 Summer,Swimming Men's 100 metres Freestyle,HUN,Hungary,Silver,1,1
7,1906 Summer,Swimming Men's 100 metres Freestyle,USA,United States,Gold,1,1
8,1908 Summer,Swimming Men's 100 metres Freestyle,HUN,Hungary,Silver,1,1
9,1908 Summer,Swimming Men's 100 metres Freestyle,SWE,Sweden,Bronze,1,1


### Medal Counting Investigation

The `Medal` column contains athlete-level records rather than a direct list of official medal awards. Therefore, counting every non-null medal row as one medal can overestimate the actual number of medals awarded.

The investigation identified the following patterns:

1. **Individual events:** Each medal-winning athlete generally has one record representing one medal award.
2. **Team events:** The same medal is repeated across the members of a medal-winning team. For example, the 1908 Men's Gymnastics Team All-Around contains 38 Swedish athlete records for one gold medal.
3. **Pairs and doubles events:** Multiple athlete records can represent one medal award. The number of records is not necessarily two. For example, Rowing Men's Coxed Pairs has three athlete records per medal-winning crew because the crew includes a coxswain.
4. **Country attribution:** `NOC` should be used to identify the country or Olympic delegation. The `Team` column may contain historical club, boat, or team-entry names rather than country names.
5. **Historical and tied results:** Different teams or athletes may receive medals in the same event. These awards must be preserved rather than merged indiscriminately.

**Cleaning decision:** we will not calculate official medal totals by simply counting non-null values in `Medal`. we will not remove repeated team-member records, since they represent valid athlete participation. Any medal-award aggregation must distinguish individual events from team events and preserve separate team entries and ties.

In [31]:
# Check repeated athlete-event combinations after exact duplicate removal

repeated_athlete_events = df_clean[
    df_clean.duplicated(
        subset=["ID", "Games", "Event"],
        keep=False
    )
].sort_values(["ID", "Games", "Event"])

print("Repeated athlete-event rows:",
      len(repeated_athlete_events))

repeated_athlete_events[
    ["ID", "Name", "Games", "Event", "Team", "NOC", "Medal"]
].head(20)

Repeated athlete-event rows: 184


,ID,Name,Games,Event,Team,NOC,Medal
4017,2317,Auguste Albert,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,FRA,Silver
4018,2317,Auguste Albert,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,FRA,Bronze
10910,5959,Louis Albert Auguste-Dormeuil (Dormeuil-),1900 Summer,Sailing Mixed 0.5-1 Ton,Pettit-Poucet-6,FRA,NaN
10911,5959,Louis Albert Auguste-Dormeuil (Dormeuil-),1900 Summer,Sailing Mixed 0.5-1 Ton,Carabinier-15,FRA,Gold
16118,8683,Jacques Baudrier,1900 Summer,Sailing Mixed 0.5-1 Ton,Crabe II-1,FRA,Silver
16119,8683,Jacques Baudrier,1900 Summer,Sailing Mixed 0.5-1 Ton,Crabe II-4,FRA,Bronze
16120,8683,Jacques Baudrier,1900 Summer,Sailing Mixed 1-2 Ton,Nina Claire-2,FRA,Bronze
16121,8683,Jacques Baudrier,1900 Summer,Sailing Mixed 1-2 Ton,Nina Claire-2,FRA,NaN
16116,8683,Jacques Baudrier,1900 Summer,Sailing Mixed Open,Crabe II-12,FRA,NaN
16117,8683,Jacques Baudrier,1900 Summer,Sailing Mixed Open,Nina Claire-30,FRA,NaN


In [32]:
repeated_groups = (
    repeated_athlete_events
    .groupby(["ID", "Games", "Event"])
    .agg(
        record_count=("ID", "size"),
        different_teams=("Team", "nunique"),
        different_medals=("Medal", "nunique"),
        medals=("Medal", lambda x: list(x.dropna().unique()))
    )
    .reset_index()
)

print("Repeated groups:", len(repeated_groups))

print("\nGroups with multiple teams:",
      (repeated_groups["different_teams"] > 1).sum())

print("Groups with multiple medal values:",
      (repeated_groups["different_medals"] > 1).sum())

print("Groups with multiple records but same team and medal:",
      (
          (repeated_groups["different_teams"] == 1) &
          (repeated_groups["different_medals"] <= 1)
      ).sum())

Repeated groups: 89

Groups with multiple teams: 20
Groups with multiple medal values: 21
Groups with multiple records but same team and medal: 55


In [33]:
multiple_medal_groups = repeated_groups[
    repeated_groups["different_medals"] > 1
]

multiple_medal_details = repeated_athlete_events.merge(
    multiple_medal_groups[["ID", "Games", "Event"]],
    on=["ID", "Games", "Event"],
    how="inner"
)

multiple_medal_details[
    ["ID", "Name", "Games", "Event", "Team", "NOC", "Medal"]
].sort_values(["ID", "Event", "Team"])

,ID,Name,Games,Event,Team,NOC,Medal
0,2317,Auguste Albert,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,FRA,Silver
1,2317,Auguste Albert,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,FRA,Bronze
2,8683,Jacques Baudrier,1900 Summer,Sailing Mixed 0.5-1 Ton,Crabe II-1,FRA,Silver
3,8683,Jacques Baudrier,1900 Summer,Sailing Mixed 0.5-1 Ton,Crabe II-4,FRA,Bronze
4,26557,Bernard Alexandre Georges Edmond de Pourtals,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Gold
5,26557,Bernard Alexandre Georges Edmond de Pourtals,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Silver
6,26559,Hlne de Pourtals (Barbey-),1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Gold
7,26559,Hlne de Pourtals (Barbey-),1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Silver
8,26560,Hermann Alexandre de Pourtals,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Gold
9,26560,Hermann Alexandre de Pourtals,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,SUI,Silver


### Investigation of Multiple Medal Values

Further inspection identified 21 athlete–Games–Event groups containing multiple distinct non-null medal values.

Examples include:
- 1900 Sailing events, where some athletes appear with different medals, sometimes for the same boat and sometimes for different boat entries.
- 1948 Art Competitions, where one athlete appears with Silver, Bronze, and a missing medal value in the same event.

These cases may reflect historical dataset inconsistencies or unusual competition records. The available data alone is insufficient to determine the correct medal for each case.

**Cleaning decision:** These records are retained and flagged for further validation. No medal values are overwritten, and no records are deleted based solely on repeated athlete–event combinations.

## Member 2-Data Cleaning — Conclusion

- Removed **1,385 exact duplicate rows** using `drop_duplicates()`.
- Identified **184 rows across 89 repeated athlete–event groups**. These were retained because they may represent legitimate historical participation.
- Investigated conflicting medal values, particularly in historical Sailing and Art Competitions records. Retained them pending validation.
- Investigated missing and extreme Age, Height, and Weight values. Missing values were preserved without imputation, and unusual values were not removed without evidence of invalidity.
- Preserved valid athlete participation records to avoid losing historical information.

**Final decision:** Only confirmed exact duplicates were removed. Other unusual or repeated records were investigated and retained when their validity could not be disproved.